# Model B: fine-tuning DeBERTa on headline pairs

This notebook fine-tunes `microsoft/deberta-v3-base` (184M parameters) as a **cross-encoder**: it reads both headlines together as one input, `[CLS] headline A [SEP] headline B [SEP]`, and predicts whether A gets more clicks than B.

**Before you start:** Runtime, then Change runtime type, then **T4 GPU**, then Save. Then run each cell in order (Shift+Enter).

Runs in about 15 minutes on a T4.

## 1. Check the GPU
You should see a Tesla T4. If you see an error, the runtime is not set to GPU.

In [2]:
!nvidia-smi

Wed Nov 18 11:43:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install packages
Colab already has PyTorch and transformers. DeBERTa's tokenizer also needs `sentencepiece`.

In [3]:
!pip install -q sentencepiece protobuf

## 3. Upload the data
Run this cell, click **Choose files**, and pick `data/processed/pairs_significant.csv` from your project folder.

In [4]:
from google.colab import files
uploaded = files.upload()
assert 'pairs_significant.csv' in uploaded, 'Upload the file named pairs_significant.csv'

Saving pairs_significant.csv to pairs_significant.csv


## 4. Settings
All hyperparameters in one place. The seed makes the run reproducible.

In [5]:
import random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

CONFIG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_length": 64,        # both headlines together; Upworthy headlines are short
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "epochs": 4,
    "warmup_ratio": 0.1,
    "seed": 42,
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int) -> None:
    """Fix every source of randomness so a rerun gives the same result."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])
print("Device:", DEVICE)

Device: cuda


## 5. Load the pairs
Same train/validation split as on your laptop.

During training every pair is also added **swapped** (B, A with the label flipped), so the model cannot learn a shortcut like "A usually wins".

In [6]:
pairs = pd.read_csv("pairs_significant.csv")
train_df = pairs[pairs["split"] == "train"].reset_index(drop=True)
val_df = pairs[pairs["split"] == "val"].reset_index(drop=True)
# The test split stays untouched until the final evaluation.
print(f"Train pairs: {len(train_df)}, validation pairs: {len(val_df)}")


class PairDataset(Dataset):
    """Each item is (headline_a, headline_b, label).

    With swap=True every pair is also added the other way round with the
    label flipped, so the model cannot learn a preference for position A or B.
    """

    def __init__(self, df: pd.DataFrame, swap: bool):
        a, b, y = list(df["headline_a"]), list(df["headline_b"]), list(df["label"])
        if swap:
            a, b, y = a + b, b + a, y + [1 - v for v in y]
        self.a, self.b, self.y = a, b, y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.a[i], self.b[i], self.y[i]


def make_collate(tokenizer, max_length: int):
    """Tokenise a batch: the two headlines go in as one input, '[CLS] A [SEP] B [SEP]'."""
    def collate(batch):
        a, b, y = zip(*batch)
        enc = tokenizer(list(a), list(b), truncation=True, max_length=max_length,
                        padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor(y)
        return enc
    return collate

Train pairs: 3212, validation pairs: 648


## 6. Load DeBERTa
The first time, this downloads the model (about 700 MB). A warning that some weights are "newly initialized" is expected: that is the classification layer we are about to train.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"], num_labels=2, dtype=torch.float32).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {CONFIG['model_name']}, {n_params / 1e6:.0f}M parameters")

collate = make_collate(tokenizer, CONFIG["max_length"])
train_loader = DataLoader(PairDataset(train_df, swap=True), batch_size=CONFIG["batch_size"],
                          shuffle=True, collate_fn=collate,
                          generator=torch.Generator().manual_seed(CONFIG["seed"]))


@torch.no_grad()
def predict_proba(df: pd.DataFrame) -> np.ndarray:
    """P(headline_a wins), averaged over both orders (A,B) and (B,A) for symmetry."""
    model.eval()
    def run(a_col, b_col):
        ds = PairDataset(df.rename(columns={a_col: "headline_a", b_col: "headline_b"}), swap=False)
        probs = []
        for batch in DataLoader(ds, batch_size=64, collate_fn=collate):
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            probs.append(torch.softmax(model(**batch).logits.float(), dim=-1)[:, 1].cpu())
        return torch.cat(probs).numpy()
    p_ab = run("headline_a", "headline_b")
    p_ba = run("headline_b", "headline_a")
    return (p_ab + (1 - p_ba)) / 2

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier

Model: microsoft/deberta-v3-base, 184M parameters


## 7. Train
AdamW with a linear learning-rate schedule and warm-up (lecture 5). After each epoch we measure validation accuracy and keep the best epoch.

At prediction time we score both orders, (A, B) and (B, A), and average them, so swapping the headlines always flips the answer.

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"],
                              weight_decay=CONFIG["weight_decay"])
total_steps = len(train_loader) * CONFIG["epochs"]
scheduler = get_linear_schedule_with_warmup(optimizer, int(CONFIG["warmup_ratio"] * total_steps), total_steps)

history, best_acc, best_state = [], 0.0, None
start = time.time()
for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running_loss = 0.0
    for step, batch in enumerate(train_loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        running_loss += loss.item()
    val_probs = predict_proba(val_df)
    val_acc = ((val_probs > 0.5).astype(int) == val_df["label"].to_numpy()).mean()
    history.append({"epoch": epoch, "train_loss": running_loss / len(train_loader), "val_accuracy": val_acc})
    print(f"Epoch {epoch}: train loss {running_loss / len(train_loader):.4f}, "
          f"validation accuracy {val_acc:.3f}  ({(time.time() - start) / 60:.1f} min)")
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest validation accuracy: {best_acc:.3f}")

Epoch 1: train loss 0.6938, validation accuracy 0.685  (2.0 min)
Epoch 2: train loss 0.3349, validation accuracy 0.736  (4.0 min)
Epoch 3: train loss 0.0725, validation accuracy 0.722  (5.9 min)
Epoch 4: train loss 0.0244, validation accuracy 0.725  (7.9 min)

Best validation accuracy: 0.736


In [9]:
# 1. Is the model really in 32-bit now?
print("Model dtype:", next(model.parameters()).dtype)

# 2. Reload cleanly in 32-bit and watch the first 30 training steps
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"], num_labels=2, dtype=torch.float32).to(DEVICE)
print("Reloaded dtype:", next(model.parameters()).dtype)

set_seed(CONFIG["seed"])
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])
model.train()
for step, batch in enumerate(train_loader, 1):
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    loss = model(**batch).loss
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    opt.zero_grad()
    print(f"step {step:2d}  loss {loss.item():.4f}  grad norm {grad_norm.item():.2f}")
    if step == 30 or torch.isnan(loss):
        break

Model dtype: torch.float16


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier

Reloaded dtype: torch.float32
step  1  loss 0.6307  grad norm 3.30
step  2  loss 0.7106  grad norm 1.00
step  3  loss 0.7475  grad norm 2.41
step  4  loss 0.7346  grad norm 1.77
step  5  loss 0.7017  grad norm 0.77
step  6  loss 0.7064  grad norm 1.39
step  7  loss 0.7044  grad norm 1.45
step  8  loss 0.6900  grad norm 0.77
step  9  loss 0.6978  grad norm 0.91
step 10  loss 0.7055  grad norm 1.23
step 11  loss 0.7072  grad norm 0.71
step 12  loss 0.6910  grad norm 2.28
step 13  loss 0.7137  grad norm 0.87
step 14  loss 0.6925  grad norm 3.74
step 15  loss 0.7092  grad norm 1.15
step 16  loss 0.6819  grad norm 1.66
step 17  loss 0.6893  grad norm 3.08
step 18  loss 0.6885  grad norm 0.99
step 19  loss 0.7034  grad norm 0.71
step 20  loss 0.7036  grad norm 0.93
step 21  loss 0.6870  grad norm 0.91
step 22  loss 0.6994  grad norm 1.76
step 23  loss 0.6830  grad norm 0.84
step 24  loss 0.6948  grad norm 1.13
step 25  loss 0.6931  grad norm 1.48
step 26  loss 0.6973  grad norm 1.72
step 27 

## 8. Save and download the results
This downloads the validation predictions (for error analysis later) and the training history.

In [12]:
model.load_state_dict(best_state)
val_probs = predict_proba(val_df)
out = val_df[["test_id", "story_group", "headline_a", "headline_b", "ctr_a", "ctr_b", "label"]].copy()
out["prob_a_wins"] = val_probs
out["prediction"] = (val_probs > 0.5).astype(int)
out["correct"] = out["prediction"] == out["label"]
out.to_csv("deberta_val_predictions.csv", index=False)
pd.DataFrame(history).to_csv("deberta_history.csv", index=False)
print(f"Saved predictions. Validation accuracy (best epoch): {out['correct'].mean():.3f}")

files.download('deberta_val_predictions.csv')
files.download('deberta_history.csv')

Saved predictions. Validation accuracy (best epoch): 0.736


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>